In [1]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# Load data
decon = pd.read_csv(
    "Decon-COO-Results_Loy_Inflammatory-Syndromes/merged_normalised_results_Loy_Inflammatory-Syndromes.txt",
    sep="\t"
)
meta = pd.read_csv("meta_ALT.csv")

# Clean sample IDs
meta["sample_id_clean"] = meta["sample_id"].str.replace("_trimmed", "", regex=False)

# Merge using correct column names
merged = pd.merge(decon, meta, left_on="Sample", right_on="sample_id_clean")

# Report matched sample count
print(f"Total matched rows (sample x tool): {merged.shape[0]}")
print(f"Unique matched samples: {merged['sample_id_clean'].nunique()}")

results = []

# If there’s a column called 'Method' or 'DeconvolutionTool', group by it
group_col = "Method" if "Method" in merged.columns else "DeconvolutionTool"

for tool, df_tool in merged.groupby(group_col):
    # Check for hepatocyte column (case-insensitive)
    hepatocyte_cols = [c for c in df_tool.columns if "hepatocyte" in c.lower()]
    if not hepatocyte_cols:
        raise ValueError("No hepatocyte column found in deconvolution file.")
    hepatocyte_col = hepatocyte_cols[0]

    hepatocyte = df_tool[hepatocyte_col].astype(float)
    alt = df_tool["palt"].astype(float)

    # Spearman correlation
    rho, pval_rho = spearmanr(hepatocyte, alt)

    # Pearson correlation after log transform
    alt_log = np.log1p(alt)
    r, pval_r = pearsonr(hepatocyte, alt_log)

    results.append({
        "DeconvolutionTool": tool,
        "Spearman_rho": rho,
        "Spearman_p": pval_rho,
        "Pearson_r_logALT": r,
        "Pearson_p": pval_r
    })

    # --- Plot with linear ALT axis ---
    plt.figure(figsize=(3, 3))
    sns.set(style="white")
    sns.set_context("paper", font_scale=1.0)

    ax = sns.regplot(
        x=alt,
        y=hepatocyte,
        scatter_kws={'s': 10, 'alpha': 0.4, 'color': 'black'},
        line_kws={'color': 'crimson', 'lw': 2}
    )

    ax.set_xlabel('ALT (U/L)', fontsize=10)
    ax.set_ylabel('Hepatocyte contribution (%)', fontsize=10)
    ax.set_title(f'Hepatocytes vs ALT ({tool}) \nSpearman rho={rho:.2f}, p={pval_rho:.2e}',
                 fontsize=13)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    sns.despine()
    plt.tight_layout()
    plt.savefig(f"ALT_vs_Hepatocyte_{tool}_linear.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"ALT_vs_Hepatocyte_{tool}_linear.pdf", dpi=600, bbox_inches="tight")
    plt.close()

    # --- Plot with log ALT axis ---
    plt.figure(figsize=(3, 3))
    sns.set(style="white")
    sns.set_context("paper", font_scale=1.0)

    ax = sns.regplot(
        x=alt,
        y=hepatocyte,
        scatter_kws={'s': 30, 'alpha': 0.4, 'color': 'black'},
        line_kws={'color': 'crimson', 'lw': 2}
    )

    ax.set_xscale("log")
    ax.set_xlabel('ALT (U/L, log scale)', fontsize=10)
    ax.set_ylabel('Hepatocyte contribution (%)', fontsize=10)
    ax.set_title(f'Hepatocytes vs ALT ({tool}) \nSpearman rho={rho:.2f}, p={pval_rho:.2e}',
                 fontsize=10)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    sns.despine()
    plt.tight_layout()
    plt.savefig(f"ALT_vs_Hepatocyte_{tool}_log.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"ALT_vs_Hepatocyte_{tool}_log.pdf", dpi=600, bbox_inches="tight")
    plt.close()

# Save correlation summary
summary = pd.DataFrame(results)
summary.to_csv("ALT_Hepatocyte_correlations.csv", index=False)
print(summary)


Total matched rows (sample x tool): 252
Unique matched samples: 36
  DeconvolutionTool  Spearman_rho  Spearman_p  Pearson_r_logALT  Pearson_p
0        BayesPrism      0.606175    0.000089          0.606918   0.000087
1        CIBERSORTx      0.254725    0.133804          0.218560   0.200324
2             MuSiC      0.313795    0.062363          0.318319   0.058483
3              NNLS     -0.187294    0.274033         -0.121544   0.480094
4                QP     -0.242810    0.153608         -0.117621   0.494483
5          ReDeconv      0.363584    0.029278          0.538486   0.000704
6             nuSVR      0.106460    0.536589          0.171488   0.317286


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import seaborn as sns
import matplotlib.pyplot as plt

# Load data
decon = pd.read_csv(
    "Decon-COO-Results_Loy_Inflammatory-Syndromes/merged_normalised_results_Loy_Inflammatory-Syndromes.txt",
    sep="\t"
)
meta = pd.read_csv("meta_ALT.csv")

# Clean sample IDs
meta["sample_id_clean"] = meta["sample_id"].str.replace("_trimmed", "", regex=False)

# Merge using correct column names
merged = pd.merge(decon, meta, left_on="Sample", right_on="sample_id_clean")

# Report matched sample count
print(f"Total matched rows (sample x tool): {merged.shape[0]}")
print(f"Unique matched samples: {merged['sample_id_clean'].nunique()}")

results = []

# Determine grouping column
group_col = "Method" if "Method" in merged.columns else "DeconvolutionTool"

for tool, df_tool in merged.groupby(group_col):
    # Find intrahepatic cholangiocyte column (case-insensitive)
    chol_cols = [c for c in df_tool.columns if "intrahepatic cholangiocyte" in c.lower()]
    if not chol_cols:
        raise ValueError("No 'intrahepatic cholangiocyte' column found in deconvolution file.")
    chol_col = chol_cols[0]

    chol = df_tool[chol_col].astype(float)
    alt = df_tool["palt"].astype(float)

    # Spearman correlation
    rho, pval_rho = spearmanr(chol, alt)

    # Pearson correlation after log transform
    alt_log = np.log1p(alt)
    r, pval_r = pearsonr(chol, alt_log)

    results.append({
        "DeconvolutionTool": tool,
        "Spearman_rho": rho,
        "Spearman_p": pval_rho,
        "Pearson_r_logALT": r,
        "Pearson_p": pval_r
    })

    # --- Plot with linear ALT axis ---
    plt.figure(figsize=(8, 6))
    sns.set(style="white", context="talk")

    ax = sns.regplot(
        x=alt,
        y=chol,
        scatter_kws={'s': 30, 'alpha': 0.5, 'color': '#4C72B0'},
        line_kws={'color': 'crimson', 'lw': 2}
    )

    ax.set_xlabel('ALT (U/L)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Intrahepatic cholangiocyte contribution (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'Intrahepatic cholangiocytes vs ALT ({tool}) \nSpearman rho={rho:.2f}, p={pval_rho:.2e}',
                 fontsize=16, fontweight='bold')
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    sns.despine()
    plt.tight_layout()
    plt.savefig(f"ALT_vs_IntrahepaticCholangiocyte_{tool}_linear.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"ALT_vs_IntrahepaticCholangiocyte_{tool}_linear.pdf", dpi=300, bbox_inches="tight")
    plt.close()

    # --- Plot with log ALT axis ---
    plt.figure(figsize=(8, 6))
    sns.set(style="white", context="talk")

    ax = sns.regplot(
        x=alt,
        y=chol,
        scatter_kws={'s': 30, 'alpha': 0.5, 'color': '#4C72B0'},
        line_kws={'color': 'crimson', 'lw': 2}
    )

    ax.set_xscale("log")
    ax.set_xlabel('ALT (U/L, log scale)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Intrahepatic cholangiocyte contribution (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'Intrahepatic cholangiocytes vs ALT ({tool}) \nSpearman rho={rho:.2f}, p={pval_rho:.2e}',
                 fontsize=16, fontweight='bold')
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    sns.despine()
    plt.tight_layout()
    plt.savefig(f"ALT_vs_IntrahepaticCholangiocyte_{tool}_log.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"ALT_vs_IntrahepaticCholangiocyte_{tool}_log.pdf", dpi=300, bbox_inches="tight")
    plt.close()

# Save correlation summary
summary = pd.DataFrame(results)
summary.to_csv("ALT_IntrahepaticCholangiocyte_correlations.csv", index=False)
print(summary)


Total matched rows (sample x tool): 252
Unique matched samples: 36


/tmp/ipykernel_3261930/2437324243.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pval_rho = spearmanr(chol, alt)
/tmp/ipykernel_3261930/2437324243.py:44: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, pval_r = pearsonr(chol, alt_log)
/tmp/ipykernel_3261930/2437324243.py:40: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, pval_rho = spearmanr(chol, alt)
/tmp/ipykernel_3261930/2437324243.py:44: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, pval_r = pearsonr(chol, alt_log)


  DeconvolutionTool  Spearman_rho  Spearman_p  Pearson_r_logALT  Pearson_p
0        BayesPrism      0.145905    0.395831         -0.009027   0.958329
1        CIBERSORTx     -0.206483    0.226951         -0.214631   0.208735
2             MuSiC     -0.407764    0.013565         -0.342145   0.041099
3              NNLS           NaN         NaN               NaN        NaN
4                QP           NaN         NaN               NaN        NaN
5          ReDeconv     -0.179250    0.295541         -0.126963   0.460581
6             nuSVR     -0.060321    0.726735         -0.042943   0.803608
